Test integrating Pydantic with LlamaIndex. Use after inserting all the data with llamaindex_redis.ipynb

Load environment variables from .env file

In [1]:
import os

import nest_asyncio
from dotenv import load_dotenv

load_dotenv("../.env")
nest_asyncio.apply() # for async issues in Jupyter Notebook

Setup the embedding model

In [2]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [3]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: ]8;id=962954;https://logfire-us.pydantic.dev/iellis02/blue-horizon\https://logfire-us.pydantic.dev/iellis02/blue-horizon]8;;\

Connect to Redis Cloud

In [5]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.redis import RedisVectorStore
from redisvl.schema import IndexSchema

redis_conn_string = os.getenv("REDIS_URL")
schema = IndexSchema.from_dict(
    {
        "index": {"name": "blue_horizon", "prefix": "blue_horizon"},
        # customize fields that are indexed
        "fields": [
            # required fields for llamaindex
            {"type": "tag", "name": "id"},
            {"type": "tag", "name": "doc_id"},
            {"type": "text", "name": "text"},
            # custom vector field for bge-small-en-v1.5 embeddings
            {
                "type": "vector",
                "name": "vector",
                "attrs": {
                    "dims": 384,
                    "algorithm": "hnsw",
                    "distance_metric": "cosine",
                },
            },
        ],
    },
)
vector_store = RedisVectorStore(schema=schema, redis_url=redis_conn_string, overwrite=False)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

17:15:49 redisvl.index.index INFO   Index already exists, not overwriting.


In [6]:
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)
retriever = index.as_retriever(similarity_top_k=4)

In [7]:
retriever.retrieve("What swimming options are there?")

[NodeWithScore(node=TextNode(id_='FAQ000009', embedding=None, metadata={'category': 'amenities', 'subcategory': 'business', 'keywords': 'pool, swimming, recreation', 'last_updated': '2024-10-15'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nIs there a swimming pool?\n\nAnswer:\nYes, we have both indoor and outdoor pools open from 6:00 AM to 10:00 PM.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.6770186424260001),
 NodeWithScore(node=TextNode(id_='FAQ000010', embedding=None, metadata={'category': 'amenities', 'subcategory': 'recreation', 'keywords': 'spa, wellness, treatments', 'last_updated': '2024-01-07'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nDo you off

In [83]:
from pprint import pprint

from pydantic_ai import Agent
from pydantic_ai.settings import ModelSettings

model_settings = ModelSettings(temperature=0)

system_prompt = """You are an assistant who helps people find out information about a hotel.
Your sole job is to query the database for information about the hotel using the tool provided.
Provide the information returned from the tool that is relevant to the user's query in a
well-formatted manner. If you decide to include an item and it has a description, be
sure to include that description.

Do not offer to do anything specific for the user. After you have answered the user's
query, simply ask if there is any other information you can provide about the hotel.

Assume that prices are in dollars.

Do not mention that you are searching a database, but you may mention that you are or
have perfomed a search.

Do not provide any instructions to the user concerning the hotel that were not provided
to you.
"""

agent = Agent(
    "openai:gpt-5.1",
    system_prompt=system_prompt,
    model_settings=model_settings,
)

@agent.tool_plain
def query_hotel_info(query: str):
    """Provide information about the hotel in response to a passed-in query.

    The query should be concise and not ask for many details.
    Accesses an FAQ database, information about hotel amenities, and information about
    hotel services. Nothing else.
    """
    print(f"query={query}")
    retrieved_nodes = retriever.retrieve(query)
    pprint(retrieved_nodes)

    return [{"metadata": node.metadata, "text": node.text} for node in retrieved_nodes]

In [84]:
result = await agent.run("What dining options are there?")
print(result.output)

13:27:40.585 agent run
13:27:40.587   chat gpt-5.1
13:27:41.880   running 1 tool
13:27:41.881     running tool: query_hotel_info
query=dining options
[NodeWithScore(node=TextNode(id_='FAQ000011', embedding=None, metadata={'category': 'services', 'subcategory': 'room service', 'keywords': 'room service, dining, food', 'last_updated': '2024-04-05'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nIs room service available?\n\nAnswer:\nRoom service is available 24/7 with a full menu during restaurant hours and a limited menu overnight.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.746651649475),
 NodeWithScore(node=TextNode(id_='FAQ000007', embedding=None, metadata={'category': 'amenities', 'subcategory': 'dining', 'keywords': 'breakfast, dining, restaurant', 'last_updated': '2024-0

In [85]:
result = await agent.run("Where can I swim?")
print(result.output)

13:28:16.859 agent run
13:28:16.861   chat gpt-5.1
13:28:18.019   running 1 tool
13:28:18.020     running tool: query_hotel_info
query=swimming options including pools, spa pools, and any nearby beaches
[NodeWithScore(node=TextNode(id_='FAQ000010', embedding=None, metadata={'category': 'amenities', 'subcategory': 'recreation', 'keywords': 'spa, wellness, treatments', 'last_updated': '2024-01-07'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nDo you offer spa services?\n\nAnswer:\nOur full-service spa offers massages, facials, and body treatments. Advance booking is recommended.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.752104759216),
 NodeWithScore(node=TextNode(id_='FAQ000009', embedding=None, metadata={'category': 'amenities', 'subcategory': 'business', 'keywords': 'pool

In [86]:
result = await agent.run("I want to be pampered.")
print(result.output)

13:28:37.193 agent run
13:28:37.195   chat gpt-5.1
13:28:38.412   running 1 tool
13:28:38.412     running tool: query_hotel_info
query=spa and wellness services, massages, and any pampering or luxury experiences
[NodeWithScore(node=TextNode(id_='FAQ000010', embedding=None, metadata={'category': 'amenities', 'subcategory': 'recreation', 'keywords': 'spa, wellness, treatments', 'last_updated': '2024-01-07'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nDo you offer spa services?\n\nAnswer:\nOur full-service spa offers massages, facials, and body treatments. Advance booking is recommended.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.816629886627),
 NodeWithScore(node=TextNode(id_='AM000007', embedding=None, metadata={'category': 'Spa Services', 'price': 150, 'duration': 75, 'av

In [87]:
result = await agent.run("I want a massage.")
print(result.output)

13:29:22.811 agent run
13:29:22.813   chat gpt-5.1
13:29:24.208   running 1 tool
13:29:24.209     running tool: query_hotel_info
query=massage services
[NodeWithScore(node=TextNode(id_='FAQ000010', embedding=None, metadata={'category': 'amenities', 'subcategory': 'recreation', 'keywords': 'spa, wellness, treatments', 'last_updated': '2024-01-07'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nDo you offer spa services?\n\nAnswer:\nOur full-service spa offers massages, facials, and body treatments. Advance booking is recommended.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.794389128685),
 NodeWithScore(node=TextNode(id_='AM000003', embedding=None, metadata={'category': 'Spa Services', 'price': 280, 'duration': 90, 'availability': 'By appointment only', 'location': 'Spa & Welln

# TODO: Fix this irrelevant info

In [96]:
result = await agent.run("Can I get some escargot?")
print(result.output)

13:35:24.296 agent run
13:35:24.298   chat gpt-5.1
13:35:25.042   running 1 tool
13:35:25.043     running tool: query_hotel_info
query=escargot
[NodeWithScore(node=TextNode(id_='FAQ000001', embedding=None, metadata={'category': 'booking', 'subcategory': 'reservations', 'keywords': 'book, reserve, reservation, booking', 'last_updated': '2024-03-09'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nHow do I make a reservation?\n\nAnswer:\nYou can make a reservation through our website, mobile app, or by calling our reservation desk. We accept all major credit cards.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.5499727725979999),
 NodeWithScore(node=TextNode(id_='FAQ000018', embedding=None, metadata={'category': 'policies', 'subcategory': 'payment policy', 'keywords': 'payment, cre

Test GPT-5.1 (faster but doesn't follow instructions as well).

In [93]:
result = await agent.run("What dining options are there?")
print(result.output)

13:32:13.769 agent run
13:32:13.771   chat gpt-5.1
13:32:14.590   running 1 tool
13:32:14.591     running tool: query_hotel_info
query=dining options
[NodeWithScore(node=TextNode(id_='FAQ000011', embedding=None, metadata={'category': 'services', 'subcategory': 'room service', 'keywords': 'room service, dining, food', 'last_updated': '2024-04-05'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nIs room service available?\n\nAnswer:\nRoom service is available 24/7 with a full menu during restaurant hours and a limited menu overnight.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.746651649475),
 NodeWithScore(node=TextNode(id_='FAQ000007', embedding=None, metadata={'category': 'amenities', 'subcategory': 'dining', 'keywords': 'breakfast, dining, restaurant', 'last_updated': '2024-0